# DeepLabV3+ Training & Inference Notebook

Notebook ini untuk melatih model DeepLabV3+ pada dataset Plant Phenotyping (20 kelas) menggunakan arsitektur dari `models/`.

**Environment:** VS Code + Colab Kernel (GPU Colab)
**Dataset:** Downloaded via `data/download_dataset.py` (kagglehub)
**Model:** DeepLabV3+ dari `models/deeplab.py` dengan backbone ResNet/Xception/DRN/MobileNet
**Output:** Checkpoint `.pth.tar` di folder `experiments/`

## 0. Setup Environment (Colab-specific)

In [1]:
import os, shutil, sys
from pathlib import Path

REPO_URL = 'https://github.com/adinmusababa/segmentasi.git'
REPO_DIR = Path('/content/segmentasi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone -b setup {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print(f'CWD: {os.getcwd()}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

REPO_PATH = REPO_DIR  # alias global untuk sel-sel berikutnya
IN_COLAB = True

Cloning into '/content/segmentasi'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 158 (delta 61), reused 148 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.57 MiB | 1.01 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/segmentasi
CWD: /content/segmentasi


In [2]:
# Download dataset
!python data/download_dataset.py

Plant Phenotyping Dataset Downloader

Downloading... (this may take a while for large datasets)
Using Colab cache for faster access to the 'plant-phenotyping-dataset' dataset.
Dataset downloaded to: /kaggle/input/plant-phenotyping-dataset
Dataset root: /kaggle/input/plant-phenotyping-dataset/Plant_Phenotyping_Datasets

Organizing dataset into data/imgs/ and data/masks/

  Processing: Plant/Ara2012
    Found 120 RGB images
    Copied: 120 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Ara2013-Canon
    Found 165 RGB images
    Copied: 165 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Tobacco
    Found 62 RGB images
    Copied: 62 image/mask pairs, Skipped: 0 (no label)

  Total: 347 image/mask pairs copied

Verifying dataset
  Images: 347
  Masks:  347
  Matched pairs: 347

Done! Dataset is ready for training.
  Images: /content/segmentasi/data/imgs  (347 files)
  Masks:  /content/segmentasi/data/masks  (347 files)
  Matched pairs: 347

To train the model, r

In [3]:
# # Colab environment setup
# import sys
# import os
# from pathlib import Path

# # Detect if running in Colab
# IN_COLAB = 'google.colab' in sys.modules
# print(f"IN_COLAB: {IN_COLAB}")

# # NOTE: Semua data (repo + dataset) di-simpan di session runtime Colab (/content),
# # yang bersifat sementara dan hilang saat runtime di-reset/disconnect.
# # Ini sesuai preferensi: TIDAK menyimpan ke Google Drive.
# # Kalau mau file hasil (dataset, checkpoint) tahan lama, simpan manual ke Drive.

# if IN_COLAB:
#     # Clone repo ke session runtime (bukan Drive)
#     REPO_PATH = Path('/content/deeplabV3-PyTorch')
#     if not REPO_PATH.exists():
#         print("Cloning repository...")
#         !git clone https://github.com/adinmusababa/deeplabV3-PyTorch.git /content/deeplabV3-PyTorch
#     os.chdir(REPO_PATH)
#     print(f"Working dir: {os.getcwd()}")
# else:
#     # Local/VS Code: assume already in repo root
#     REPO_PATH = Path.cwd()
#     while not (REPO_PATH / 'models').exists() and REPO_PATH != REPO_PATH.parent:
#         REPO_PATH = REPO_PATH.parent
#     os.chdir(REPO_PATH)
#     print(f"Working dir: {os.getcwd()}")

# # Add project root to sys.path
# if str(REPO_PATH) not in sys.path:
#     sys.path.insert(0, str(REPO_PATH))

# # Verify structure
# print("models/ exists:", (REPO_PATH / "models").exists())
# print("data/ exists:", (REPO_PATH / "data").exists())
# print("configs/ exists:", (REPO_PATH / "configs").exists())
# print("kagglehub cache di: /root/.cache/kagglehub (session temp, bukan Drive)")

## 1. Install Dependencies

In [4]:
# Install requirements
!pip install -q kagglehub pyyaml tensorboardX tqdm scikit-learn matplotlib pillow numpy torch torchvision

# Verify torch CUDA
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


## 2. Download & Organize Dataset

In [5]:
# # Run download script
# import subprocess
# result = subprocess.run([sys.executable, "data/download_dataset.py"], capture_output=True, text=True)
# print(result.stdout)
# if result.stderr:
#     print("STDERR:", result.stderr)

# # Verify
# from pathlib import Path
# imgs = list((REPO_PATH / "data" / "imgs").glob("*.png"))
# masks = list((REPO_PATH / "data" / "masks").glob("*.png"))
# print(f"Images: {len(imgs)}")
# print(f"Masks: {len(masks)}")
# if imgs:
#     print(f"First few: {[f.name for f in imgs[:5]]}")

## 3. Configure Training (EDIT HERE)

In [5]:
import torch
import yaml
from pathlib import Path

# Load base config
with open("configs/config.yml") as f:
    config = yaml.safe_load(f)

# ========== OVERRIDE FOR PLANT DATASET ==========
config["dataset"]["base_path"] = str(REPO_PATH)  # root project
config["dataset"]["dataset_name"] = "plant_phenotyping"
# Plant dataset: background(0) + 19 plant organ classes = 20
config["network"]["num_classes"] = 20
config["network"]["backbone"] = "resnet"  # pilihan: resnet, xception, drn, mobilenet
config["network"]["sync_bn"] = False  # True hanya kalau multi-GPU
config["network"]["freeze_bn"] = False
config["network"]["use_cuda"] = torch.cuda.is_available()

config["image"]["out_stride"] = 16
config["image"]["base_size"] = 513
config["image"]["crop_size"] = 513  # turunkan ke 256/320 untuk eksperimen cepat

config["training"]["workers"] = 4 if torch.cuda.is_available() else 0
config["training"]["batch_size"] = 4 if torch.cuda.is_available() else 2  # minimal 2 untuk BatchNorm
config["training"]["epochs"] = 10  # ubah sesuai kebutuhan
config["training"]["start_epoch"] = 0
config["training"]["lr"] = 0.0005
config["training"]["lr_scheduler"] = "poly"  # poly, step, cos
config["training"]["momentum"] = 0.9
config["training"]["weight_decay"] = 0.0005
config["training"]["nesterov"] = False
config["training"]["loss_type"] = "ce"  # ce atau focal
config["training"]["use_balanced_weights"] = False
config["training"]["no_val"] = False
config["training"]["val_interval"] = 1
config["training"]["train_on_subset"]["enabled"] = False  # True untuk quick test
config["training"]["train_on_subset"]["dataset_fraction"] = 0.1

# Resume training (optional)
config["training"]["weights_initialization"]["use_pretrained_weights"] = False  # True kalau mau resume
config["training"]["weights_initialization"]["restore_from"] = "./experiments/checkpoint_last.pth.tar"

config["training"]["model_best_checkpoint"]["enabled"] = True
config["training"]["model_best_checkpoint"]["out_file"] = "./experiments/checkpoint_best.pth.tar"
config["training"]["model_last_checkpoint"]["enabled"] = True
config["training"]["model_last_checkpoint"]["out_file"] = "./experiments/checkpoint_last.pth.tar"
# Saver uses ./experiments/ directory (hardcoded in utils/saver.py)

config["training"]["tensorboard"]["enabled"] = True
config["training"]["tensorboard"]["log_dir"] = "./tensorboard/"

# Seed for reproducibility
config["seed"] = 42

# Save modified config
config_path = REPO_PATH / "configs" / "config_plant.yml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config saved to: {config_path}")
print("Key settings:")
print(f"  num_classes: {config["network"]["num_classes"]}")
print(f"  backbone: {config["network"]["backbone"]}")
print(f"  batch_size: {config["training"]["batch_size"]}")
print(f"  epochs: {config["training"]["epochs"]}")
print(f"  crop_size: {config["image"]["crop_size"]}")
print(f"  use_cuda: {config["network"]["use_cuda"]}")

Config saved to: /content/segmentasi/configs/config_plant.yml
Key settings:
  num_classes: 20
  backbone: resnet
  batch_size: 4
  epochs: 10
  crop_size: 513
  use_cuda: True


## 4. Training

In [6]:
# Import Trainer
from trainers.trainer import Trainer

# Buat direktori experiments jika belum ada (untuk checkpoint)
(REPO_PATH / "experiments").mkdir(parents=True, exist_ok=True)

# checkname diperlukan oleh Trainer/Saver
config["checkname"] = "deeplab-" + str(config["network"]["backbone"])

# Initialize trainer
trainer = Trainer(config)

print(f"Starting Epoch: {trainer.config["training"]["start_epoch"]}")
print(f"Total Epochs: {trainer.config["training"]["epochs"]}")
print(f"Train loader: {len(trainer.train_loader)} batches")
print(f"Val loader: {len(trainer.val_loader)} batches")
print(f"Test loader: {len(trainer.test_loader)} batches")
print(f"Classes: {trainer.nclass}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Using poly LR Scheduler!
Starting Epoch: 0
Total Epochs: 10
Train loader: 70 batches
Val loader: 34 batches
Test loader: 34 batches
Classes: 20


In [7]:
# Run training loop
for epoch in range(trainer.config['training']['start_epoch'], trainer.config['training']['epochs']):
    trainer.training(epoch)
    if not trainer.config['training']['no_val'] and epoch % config['training']['val_interval'] == (config['training']['val_interval'] - 1):
        trainer.validation(epoch)

trainer.writer.close()
print("Training completed!")

  0%|          | 0/70 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



=>Epoches 0, learning rate = 0.0005,                 previous best = 0.0000


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)
Train loss: 0.363: 100%|██████████| 70/70 [01:07<00:00,  1.03it/s]


[Epoch: 0, numImages:   279]
Loss: 25.393


Val loss: 1.073: 100%|██████████| 34/34 [00:03<00:00,  8.92it/s]


...


=>Epoches 9, learning rate = 0.0001,                 previous best = 0.0863


Train loss: 0.184: 100%|██████████| 70/70 [01:06<00:00,  1.06it/s]


[Epoch: 9, numImages:   279]
Loss: 12.911


Val loss: 0.755: 100%|██████████| 34/34 [00:03<00:00,  9.05it/s]


Validation:
[Epoch: 9, numImages:   133]
Acc:0.7836164549150143, Acc_class:0.13094553506948353, mIoU:0.08673738845004272, fwIoU: 0.7486549086907848
Loss: 25.685
Training completed!


## 5. Load Best Model for Inference

In [8]:
# Load predictor with best checkpoint
from predictors.predictor import Predictor

checkpoint_path = './experiments/checkpoint_best.pth.tar'
if not Path(checkpoint_path).exists():
    checkpoint_path = './experiments/checkpoint_last.pth.tar'
    print(f"Best not found, using last: {checkpoint_path}")
else:
    print(f"Using best checkpoint: {checkpoint_path}")

predictor = Predictor(config, checkpoint_path=checkpoint_path)
print(f"Model loaded. Classes: {predictor.num_classes}")

Using best checkpoint: ./experiments/checkpoint_best.pth.tar
Model loaded. Classes: 20


## 6. Inference on Single Image

In [8]:
# Test on a sample image from dataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Check prerequisites
if "REPO_PATH" not in globals():
    raise RuntimeError("REPO_PATH not defined. Run Cell 2 (clone repo) first.")
if "predictor" not in globals():
    raise RuntimeError("predictor not defined. Run Cell 15 (load predictor) first.")

# Pick first image from data/imgs
imgs_dir = REPO_PATH / "data" / "imgs"
if not imgs_dir.exists():
    raise RuntimeError(f"data/imgs not found at {imgs_dir}. Run Cell 3 (download dataset) first.")

test_images = list(imgs_dir.glob("*.png"))
if not test_images:
    raise RuntimeError(f"No PNG images found in {imgs_dir}. Run Cell 3 (download dataset) first.")

test_img = str(test_images[0])
print(f"Testing on: {test_img}")
print(f"Total images available: {len(test_images)}")

image, prediction = predictor.segment_image(test_img)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image.astype(np.uint8))
axes[0].set_title("Original Image")
axes[0].axis('off')

# Prediction mask
im1 = axes[1].imshow(prediction, cmap="nipy_spectral", vmin=0, vmax=predictor.num_classes-1)
axes[1].set_title("Prediction Mask")
axes[1].axis('off')

# Overlay
overlay = image.copy()
# Create colormap
colors = np.random.RandomState(42).randint(0, 255, (predictor.num_classes, 3)).astype(np.uint8)
colors[0] = [0, 0, 0]  # background black
pred_colored = colors[prediction]
overlay = (overlay * 0.6 + pred_colored * 0.4).astype(np.uint8)
axes[2].imshow(overlay)
axes[2].set_title("Overlay (60% img + 40% mask)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Prediction shape: {prediction.shape}")
print(f"Unique classes predicted: {np.unique(prediction)}")

In [13]:
# Plot training history dari TensorBoard logs
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

TENSORBOARD_DIR = REPO_PATH / "tensorboard"

# Collect all scalars from all event files
ea = EventAccumulator(str(TENSORBOARD_DIR), size_warning=False)
ea.Reload()

tags = ea.Tags()["scalars"]
print(f"Found {len(tags)} scalar tags: {tags}")

# Build a dict of tag -> (steps, values)
data = {}
for tag in tags:
    events = ea.Scalars(tag)
    steps = [e.step for e in events]
    vals = [e.value for e in events]
    data[tag] = (steps, vals)

# Plot combined training history
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("DeepLabV3+ Training History", fontsize=16, fontweight="bold")

# 1. Training & Validation Loss
ax = axes[0, 0]
if "train/total_loss_epoch" in data:
    steps, vals = data["train/total_loss_epoch"]
    ax.plot(steps, vals, "b-", label="Train Loss", linewidth=2)
if "val/total_loss_epoch" in data:
    steps, vals = data["val/total_loss_epoch"]
    ax.plot(steps, vals, "r-", label="Val Loss", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. mIoU
ax = axes[0, 1]
if "val/mIoU" in data:
    steps, vals = data["val/mIoU"]
    ax.plot(steps, vals, "g-", label="mIoU", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("mIoU")
ax.set_title("Mean Intersection over Union")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Accuracy
ax = axes[1, 0]
if "val/Acc" in data:
    steps, vals = data["val/Acc"]
    ax.plot(steps, vals, "c-", label="Overall Acc", linewidth=2)
if "val/Acc_class" in data:
    steps, vals = data["val/Acc_class"]
    ax.plot(steps, vals, "m-", label="Mean Acc", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. FWIoU
ax = axes[1, 1]
if "val/fwIoU" in data:
    steps, vals = data["val/fwIoU"]
    ax.plot(steps, vals, "y-", label="FWIoU", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Frequency Weighted IoU")
ax.set_title("Frequency Weighted IoU")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Training history plotted successfully!")

## 7. Batch Inference on Test Set (Evaluation)

In [10]:
# Run evaluation on test set
predictor.inference_on_test_set()

inference on test set


Test loss: 0.755: 100%|██████████| 34/34 [00:04<00:00,  7.97it/s]

Accuracy:0.7836164549150143, Accuracy per class:0.13094553506948353, mean IoU:0.08673738845004272, frequency weighted IoU: 0.7486549086907848
Loss: 25.685


## 8. Batch Inference on Folder (Save Predictions)

In [11]:
# Save predictions for all images in a folder
from pathlib import Path
from tqdm import tqdm

INPUT_DIR = REPO_PATH / "data" / "imgs"
OUTPUT_DIR = REPO_PATH / "inference_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

img_files = sorted(list(INPUT_DIR.glob("*.png")))
print(f"Processing {len(img_files)} images...")

for img_path in tqdm(img_files):
    try:
        _, prediction = predictor.segment_image(str(img_path))
        out_path = OUTPUT_DIR / f"{img_path.stem}_pred.png"
        Image.fromarray(prediction.astype(np.uint8)).save(out_path)
    except Exception as e:
        print(f"Error on {img_path.name}: {e}")

print(f"Done! Results saved to: {OUTPUT_DIR}")

Processing 347 images...


  0%|          | 1/347 [00:00<00:49,  6.93it/s]

Error on ara2012_plant001.png: name 'Image' is not defined


  1%|          | 2/347 [00:00<00:48,  7.05it/s]

Error on ara2012_plant002.png: name 'Image' is not defined


...

Error on tobacco_plant060.png: name 'Image' is not defined


100%|█████████▉| 346/347 [00:49<00:00,  2.98it/s]

Error on tobacco_plant061.png: name 'Image' is not defined


100%|██████████| 347/347 [00:49<00:00,  7.00it/s]

Error on tobacco_plant062.png: name 'Image' is not defined
Done! Results saved to: /content/segmentasi/inference_results


## 9. TensorBoard (Optional)

In [14]:
# Launch TensorBoard in Colab
if IN_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir ./tensorboard --port 6006
else:
    print("Run locally: tensorboard --logdir ./tensorboard")

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 14713), started 0:00:54 ago. (Use '!kill 14713' to kill it.)

<IPython.core.display.Javascript object>

## 10. Tips & Next Steps

- **Cepatkan eksperimen:** turunkan `crop_size` ke 256/320, `epochs` ke 5-10, `train_on_subset.enabled: true`
- **Ganti backbone:** `mobilenet` atau `xception` lebih cepat dari `resnet`
- **Resume training:** set `weights_initialization.use_pretrained_weights: true` dan `start_epoch`
- **Class weights:** enable `use_balanced_weights: true` untuk dataset tidak seimbang
- **Multi-GPU:** set `sync_bn: true` dan `use_cuda: true` (Colab Pro+ dengan multi-GPU)
- **Checkpoint format:** `.pth.tar` standar PyTorch, bisa di-load di `main.py` atau script custom